# 목표

연말정산 신고 안내 문서 활용 RAG 시스템 구현

In [41]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "14"

ROOT_DIR = os.getcwd()


try:
    from google.colab import drive, userdata
    IS_COLAB_MODE = True
    print("코랩 모드")

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"))
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-v{PROJECT_NUM}"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=True)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "2024년+원천징수의무자를+위한+연말정산+신고안내.pdf")

로컬 모드


### 텍스트 데이터

In [42]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

for text in docs:
    text.metadata = {"page": text.metadata["page"]}

In [43]:
# 문서 구조 기반 페이지 정리

useless_page_list = [0, 1, 15]
abstract_page_list = list(range(2, 10))
index_page_list = [10, 11, 12, 13] # content에 붙어 있는 페이지에 16 더해야 pdf 기준 페이지임.

DOCS_ABSTRACT = [docs[i] for i in abstract_page_list]
DOCS_INDEX = [docs[i] for i in index_page_list]
DOCS_CONCRETE = [docs[16:]]

In [49]:
DOCS_ABSTRACT

[Document(metadata={'page': 2, 'chapter_title': '국세청 연말정산 서비스'}, page_content='이용자 서비스 내 용 접근 경로\n국세청 홈페이지(www.nts.go.kr) \ue3fc\n원천징수(연말정산)안내 홈페이지\n국세신고안내 \ue3fc 개인 또는 법인 \ue3fc 연말정산\n국세법령정보시스템\n공통 연말정산 관련 질의회신 및 판례 조회\n(www.hometax.go.kr \ue3fc 법령정보)\n국세청 국세상담센터\n인터넷 상담 및 회신\n(www.hometax.go.kr \ue3fc 상담/제보)\n홈택스 \ue3fc 장려금·연말정산·기부금 \ue3fc\n연말정산 소득·세액공제 자료 조회 연말정산간소화\n[ 안내 ] 126-내선1-5번\n소득공제\n자료조회 현금영수증\n(홈택스 \ue3fc 전자(세금)계산서·현금영수증·\n현금영수증 발행금액 조회\n신용카드 \ue3fc 현금영수증(근로자·소비자))\n[ 안내 ] 126-내선1-1번\n홈택스 \ue3fc 장려금·연말정산·기부금 \ue3fc\n공제신고서\n간소화자료 선택 후 신고서 자동 반영 편리한 연말정산\n작성\n[ 안내 ] 126-내선1-5번\n홈택스 \ue3fc 장려금·연말정산·기부금 \ue3fc\n작 성된 공제신고서 및 증명자료\n간편제출 편리한 연말정산\n온라인 제출\n[ 안내 ] 126-내선1-5번\n근로자\n과거 원천징수 영수증(지급명세서)조회\n* ’ 19년~’24년 및 ’25년 중 제출한\n’25년 귀속분(중도퇴사자 등)은 조회 가능 홈택스 \ue3fc My홈택스 \ue3fc 연말정산 \ue3fc\n신고결과\n* ’ 25.8월 수시오픈 이후부터는 지급명세서 등 제출내역\n조회\n’19년 귀속 확인 불가 [ 안내 ] 126-내선1-3번\n제 출된 연말정산 신고사항은 제출\n다음날부터 조회 가능\n「근로자를 위한 연말정산 안내」 책자\n안내책자 국세청 홈페이지(www.nts.go.kr) \ue3fc\n「근로자를 위한 연말정산」 동영상\n및 영상 

In [48]:
import pdfplumber

with pdfplumber.open(PDF_PATH) as pdf:
    for page in DOCS_ABSTRACT:
        page_num = page.metadata["page"] 

        head = pdf.pages[page_num].within_bbox((0, 0, 538, 130))
        tail = pdf.pages[page_num].within_bbox((0, 131, 538, 737))

        if head.extract_text():
            target_text = head.extract_text().replace("\n", " ")

        page.metadata["chapter_title"] = target_text
        page.page_content = tail.extract_text()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chapter_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
    separators=[r"(?:\n|^)(?:제\s*\d+장.+)"],
    is_separator_regex=True,
    keep_separator=True
)

chapters = chapter_splitter.split_documents(DOCS_ABSTRACT)

### 표 데이터

In [30]:
from img2table.document import PDF
from img2table.ocr import TesseractOCR

pdf = PDF(
    PDF_PATH, 
    detect_rotation=False,
    pdf_text_extraction=True
)

ocr = TesseractOCR(n_threads=1, lang="eng")

TABLES_BY_PAGE = pdf.extract_tables(
    ocr=ocr,
    implicit_rows=False,
    implicit_columns=False,
    borderless_tables=False,
    min_confidence=40
)

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.54 : libtiff 4.7.1 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.5 zlib/1.2.12 liblzma/5.8.2 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.1 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.67.1


In [31]:
# 표 데이터 - 메타데이터 title 보정

TABLE_TITLE_LIST = list()
for page_num, tables in TABLES_BY_PAGE.items():
    for table in tables:
        if table and table.title:
            if len(table.title) > 25:
                table.title = None
            else:
                TABLE_TITLE_LIST.append(table.title)

In [ ]:
# 표 데이터 metadata에 merge
import pandas as pd

def trim_table(df: pd.DataFrame):
    df.columns = df.iloc[0]
    df = df[1:]
    df.reset_index(drop=True, inplace=True)

    return df


to_delete_list = list()

for page_num, tables in TABLES_BY_PAGE.items():
    if tables:
        docs[page_num].metadata["table"] = {
            f"{page_num}.{i}": trim_table(table.df) for i, table in enumerate(tables)
        }

        for table in tables:
            table = trim_table(table.df)

    else:
        docs[page_num].metadata["table"] = None
        to_delete_list.append(page_num)

for page_num in to_delete_list:
    del TABLES_BY_PAGE[page_num]

In [33]:
TABLES_BY_PAGE[18][0].title

pdf 육안 확인 + df로 바꿔보니 여간 복잡한 게 아니다.

1. 병합된 셀이 많다.
2. 특수문자(O) 같은 게 씹히는 경우가 있다.
3. 타이틀을 일일이 달아줘야 할 것 같다.
4. 표에서 확인할 수 있는 정보는 표를 참고하라고 따로 명령해야 할 듯.

In [34]:
"https://huggingface.co/microsoft/tapex-base-finetuned-wikisql"

'https://huggingface.co/microsoft/tapex-base-finetuned-wikisql'

## RAG 설계

0. 문장 데이터와 표 데이터로 나눈다.
1. 문장 데이터는 OPENAI 모델이 진행.
2. 표 데이터는 https://huggingface.co/microsoft/tapex-base-finetuned-wikisql
3. 허위 정보를 알려줘서는 안 되니, 모르는 건 모른다고 하자.
4. 참고한 페이지 정보를 같이 출력해주어 유저가 더블 체크할 수 있도록 하자.